# Prepare OAM-TCD Assets

This notebook maintains the **original, category-collapsed assets** used by the Unified-Prompt SAM3 baseline. Prompt-granularity is implemented separately in `Core/prompt_granularity.py`; use its `prepare` action for the derived annotation views. Do not change this notebook into a two-category export or rerun the original exporter over the locked assets.

Reuse existing assets with all `FORCE_*` switches false; nothing here cleans up or relocates them. The locked recipe retains all 4,169 TRAIN images and the 439 TEST images for monitoring and reporting, with last-epoch checkpoints and no validation carve.

For Mask R-CNN the SAM3 checkpoint download is unnecessary; the same original COCO export remains available. The prompt-granularity view shares the unchanged image files and preserves geometry, IDs, original category IDs, and crowd flags.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "Core").is_dir():
    CODEBASE = cwd
elif cwd.name == "Notebook" and (cwd.parent / "Core").is_dir():
    CODEBASE = cwd.parent
else:
    candidates = [p for p in [cwd, *cwd.parents] if (p / "Codebase" / "Core").is_dir()]
    if not candidates:
        raise RuntimeError(f"Could not locate Codebase/Core from cwd={cwd}")
    CODEBASE = candidates[0] / "Codebase"

CORE = CODEBASE / "Core"
if str(CORE) not in sys.path:
    sys.path.insert(0, str(CORE))

from project_workflows import create_asset_workflow

In [ ]:
FORCE_DOWNLOAD_ASSETS = False
FORCE_EXPORT_FULL = False

assets = create_asset_workflow(
    force_download_assets=FORCE_DOWNLOAD_ASSETS,
    force_export_full=FORCE_EXPORT_FULL,
)
assets.print_summary()

## 1. Download SAM3 Checkpoint

Writes `runtime/checkpoints/asset_manifest.json`. Not needed for Mask R-CNN training but harmless to run.

In [ ]:
assets.ensure_checkpoint()

## 2. Export OAM-TCD Instance COCO Dataset

The cleaned, category-collapsed export is consumed by `experiment_pipeline.ipynb`. Keep `FORCE_EXPORT_FULL=False` when using existing assets. A missing or inconsistent export requires investigation before regeneration; the prompt-granularity arm must not overwrite it.

In [ ]:
export_report = assets.ensure_export()

## 3. Verify the Locked Split Selection

The baseline trains on `train_annotations.coco.json` and monitors `test_annotations.coco.json`. Final reporting uses the last-epoch checkpoint under the locked tiled protocol. No validation carve is created or selected here. TEST informed monitoring, so it must not be described as an untouched model-selection holdout.

In [ ]:
from config import TRAIN_ANN_FILENAME, VAL_ANN_FILENAME, TEST_ANN_FILENAME

assert TRAIN_ANN_FILENAME == "train_annotations.coco.json"
assert VAL_ANN_FILENAME == TEST_ANN_FILENAME == "test_annotations.coco.json"
print("TRAIN:", TRAIN_ANN_FILENAME)
print("Monitoring / reporting:", VAL_ANN_FILENAME)
print("No validation carve is created. The category-collapsed export remains unchanged.")

## 4. Final Asset Record

In [ ]:
import json

record = assets.asset_record()
print(json.dumps(record, indent=2))
assert all((
    record["asset_manifest_exists"],
    record["sam3_checkpoint_exists"],
    record["sam3_config_exists"],
    record["export_report_exists"],
    record["production_export_ready"],
))

## 5. Per-image GT Count Distribution

Bucketed histogram of ground-truth instance counts per image for both `train` and `test` splits of the exported instance COCO. This characterises how often individual tiles exceed the model's fixed query budget and informs whether the decoder query cap is an active recall bottleneck.

In [ ]:
GT_COUNT_REPORT = assets.write_gt_count_distribution()